# DSA 210 - Phase 4: Machine Learning Methods
## Music Listening Behavior & Weather Analysis

**Deha Ozan | Spring 2026**

---

### Overview

In this phase, four supervised machine learning methods are applied to the `music_weather_daily.csv` dataset (900 daily observations, Oct 2023 - present) to model the relationship between Istanbul weather and personal Apple Music listening behavior.

| # | Method | Task | Target | Features |
|---|--------|------|--------|----------|
| 1 | Linear Regression | Regression | `total_listen_min` | Weather features |
| 2 | Logistic Regression | Classification | `is_rainy` | Listening features |
| 3 | K-Nearest Neighbors | Regression | `total_listen_min` | Weather + time + listening |
| 4 | Decision Tree | Classification | `condition` | Listening + time features |

Methods 1 and 3 ask whether weather predicts listening time. Methods 2 and 4 reverse the question: does how I listen predict the weather? This symmetric design tests whether the relationship holds in both directions.

## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    mean_absolute_error, r2_score,
    accuracy_score, classification_report, confusion_matrix
)

plt.rcParams['figure.dpi'] = 120
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
print('All libraries loaded.')

In [ ]:
df = pd.read_csv('music_weather_daily.csv', parse_dates=['date'])
print(f'Shape: {df.shape}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
df.head()

In [ ]:
# Feature distributions overview
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold')

for i, col in enumerate(['total_listen_min', 'n_plays', 'skip_rate']):
    axes[0, i].hist(df[col], bins=40, color=PALETTE[0], alpha=0.8, edgecolor='white')
    axes[0, i].set_title(col)

for i, col in enumerate(['temp_mean', 'precipitation', 'sunshine_hours']):
    axes[1, i].hist(df[col], bins=40, color=PALETTE[1], alpha=0.8, edgecolor='white')
    axes[1, i].set_title(col)

plt.tight_layout()
plt.savefig('fig_phase4_distributions.png', bbox_inches='tight')
plt.show()

---
## 1. Linear Regression

**Question:** Can daily weather conditions linearly predict how many minutes I listen to music?

**Features:** `temp_mean`, `temp_max`, `precipitation`, `sunshine_hours`  
**Target:** `total_listen_min`

Linear regression is the baseline model. A low R2 here would indicate that weather alone does not linearly explain listening duration.

In [ ]:
X_lr = df[['temp_mean', 'temp_max', 'precipitation', 'sunshine_hours']]
y_lr = df['total_listen_min']

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lr, y_lr, test_size=0.2, random_state=42
)

lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train_lr)
y_pred_lr = lr_model.predict(X_test_lr)

lr_r2  = r2_score(y_test_lr, y_pred_lr)
lr_mae = mean_absolute_error(y_test_lr, y_pred_lr)

print('Linear Regression: total_listen_min ~ weather features')
print(f'  R2  : {lr_r2:.4f}')
print(f'  MAE : {lr_mae:.4f} minutes')
print()
print('Feature coefficients:')
for feat, coef in zip(X_lr.columns, lr_model.coef_):
    print(f'  {feat:<22} {coef:+.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Linear Regression: Weather to Listening Duration', fontsize=13, fontweight='bold')

# Actual vs Predicted
axes[0].scatter(y_test_lr, y_pred_lr, alpha=0.5, color=PALETTE[0], edgecolors='white', linewidth=0.4, s=40)
lims = [0, max(y_test_lr.max(), y_pred_lr.max()) + 10]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual (minutes)')
axes[0].set_ylabel('Predicted (minutes)')
axes[0].set_title(f'Actual vs Predicted  (R2 = {lr_r2:.3f})')
axes[0].legend()

# Residuals
residuals_lr = y_test_lr - y_pred_lr
axes[1].scatter(y_pred_lr, residuals_lr, alpha=0.5, color=PALETTE[1], edgecolors='white', linewidth=0.4, s=40)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted (minutes)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.savefig('fig_phase4_linear_regression.png', bbox_inches='tight')
plt.show()

In [ ]:
# Coefficient chart
fig, ax = plt.subplots(figsize=(7, 4))
coefs = pd.Series(lr_model.coef_, index=X_lr.columns)
colors = [PALETTE[2] if c > 0 else PALETTE[3] for c in coefs]
coefs.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient value')
ax.set_title('Feature Coefficients', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_phase4_lr_coefficients.png', bbox_inches='tight')
plt.show()

### Linear Regression Interpretation

The model achieves R2 = 0.05 and MAE = 59 minutes. Weather features alone explain almost none of the variance in daily listening time. The residual plot shows large uniform scatter, indicating missing variables rather than model misspecification.

The coefficient signs are interesting: precipitation has a negative coefficient (more rain = slightly less listening), contradicting the intuitive assumption. Sunshine hours show a small positive effect.

**Key takeaway:** Weather does not linearly predict listening time. This motivates more flexible methods.

---
## 2. Logistic Regression

**Question:** Does listening behavior allow us to predict whether it was a rainy day?

**Features:** `n_plays`, `total_listen_min`, `skip_rate`, `fraction_complete`  
**Target:** `is_rainy` (binary)

This reverses the direction of analysis. If the model cannot beat the majority-class baseline (71% not rainy), then listening patterns carry no useful signal about rain.

In [ ]:
X_log = df[['n_plays', 'total_listen_min', 'skip_rate', 'fraction_complete']]
y_log = df['is_rainy'].astype(int)

X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_log, y_log, test_size=0.2, random_state=42
)

# Standardise - logistic regression is scale-sensitive
scaler_log = StandardScaler()
X_train_logs = scaler_log.fit_transform(X_train_log)
X_test_logs  = scaler_log.transform(X_test_log)

log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_logs, y_train_log)
y_pred_log = log_model.predict(X_test_logs)

log_acc  = accuracy_score(y_test_log, y_pred_log)
baseline = y_test_log.value_counts(normalize=True).max()

print('Logistic Regression: is_rainy ~ listening behavior')
print(f'  Accuracy         : {log_acc:.4f}')
print(f'  Majority baseline: {baseline:.4f}')
print(f'  Lift over baseline: {log_acc - baseline:+.4f}')
print()
print(classification_report(y_test_log, y_pred_log,
      target_names=['Not Rainy', 'Rainy'], zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Logistic Regression: Listening Behavior to Rainy Day', fontsize=13, fontweight='bold')

# Confusion matrix
cm = confusion_matrix(y_test_log, y_pred_log)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Rainy', 'Rainy'], yticklabels=['Not Rainy', 'Rainy'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# Accuracy vs baseline
bars = axes[1].bar(['Majority Baseline', 'Logistic Regression'],
                   [baseline, log_acc],
                   color=[PALETTE[3], PALETTE[0]], width=0.5, edgecolor='white')
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Model vs Majority-Class Baseline')
for bar, val in zip(bars, [baseline, log_acc]):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.3f}',
                 ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_phase4_logistic_regression.png', bbox_inches='tight')
plt.show()

### Logistic Regression Interpretation

Accuracy = 70.6%, virtually identical to the 71% majority baseline. The confusion matrix shows the model almost entirely predicts 'not rainy', with near-zero recall for rainy days.

This is consistent with the Phase 3 Mann-Whitney results: listening behavior distributions on rainy vs non-rainy days overlap substantially, making classification very difficult.

**Key takeaway:** Listening patterns carry no discriminative signal about rain. The relationship is not symmetric.

---
## 3. K-Nearest Neighbors (KNN) Regression

**Question:** Can we do better at predicting listening time by using all features and a nonlinear method?

**Features:** `temp_mean`, `precipitation`, `sunshine_hours`, `n_plays`, `skip_rate`, `day_of_week`, `month`  
**Target:** `total_listen_min`

KNN regression predicts a value by averaging the k most similar historical days. Features are standardised since KNN is distance-based. k is tuned across 6 values.

In [ ]:
X_knn = df[['temp_mean', 'precipitation', 'sunshine_hours',
            'n_plays', 'skip_rate', 'day_of_week', 'month']]
y_knn = df['total_listen_min']

X_train_knn, X_test_knn, y_train_knn, y_test_knn = train_test_split(
    X_knn, y_knn, test_size=0.2, random_state=42
)

scaler_knn = StandardScaler()
X_train_knns = scaler_knn.fit_transform(X_train_knn)
X_test_knns  = scaler_knn.transform(X_test_knn)

k_values = [1, 3, 5, 10, 20, 50]
knn_results = []
for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train_knns, y_train_knn)
    preds = knn.predict(X_test_knns)
    knn_results.append({'k': k, 'R2': r2_score(y_test_knn, preds),
                        'MAE': mean_absolute_error(y_test_knn, preds)})

knn_df = pd.DataFrame(knn_results)
print('KNN k-tuning results:')
print(knn_df.to_string(index=False))
best_k = int(knn_df.loc[knn_df['R2'].idxmax(), 'k'])
print(f'Best k: {best_k}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('KNN Regression: k Tuning', fontsize=13, fontweight='bold')

axes[0].plot(knn_df['k'], knn_df['R2'], 'o-', color=PALETTE[0], linewidth=2, markersize=7)
axes[0].set_xlabel('k')
axes[0].set_ylabel('R2')
axes[0].set_title('R2 by k')
axes[0].grid(alpha=0.3)

axes[1].plot(knn_df['k'], knn_df['MAE'], 'o-', color=PALETTE[1], linewidth=2, markersize=7)
axes[1].set_xlabel('k')
axes[1].set_ylabel('MAE (minutes)')
axes[1].set_title('MAE by k')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig_phase4_knn_tuning.png', bbox_inches='tight')
plt.show()

In [ ]:
# Best model
knn_best = KNeighborsRegressor(n_neighbors=best_k)
knn_best.fit(X_train_knns, y_train_knn)
y_pred_knn = knn_best.predict(X_test_knns)

knn_r2  = r2_score(y_test_knn, y_pred_knn)
knn_mae = mean_absolute_error(y_test_knn, y_pred_knn)

print(f'Best KNN (k={best_k}):  R2={knn_r2:.4f}  MAE={knn_mae:.4f} min')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'KNN Regression (k={best_k}): All Features to Listening Duration',
             fontsize=13, fontweight='bold')

axes[0].scatter(y_test_knn, y_pred_knn, alpha=0.5, color=PALETTE[0],
                edgecolors='white', linewidth=0.4, s=40)
lims = [0, max(y_test_knn.max(), y_pred_knn.max()) + 10]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual (minutes)')
axes[0].set_ylabel('Predicted (minutes)')
axes[0].set_title(f'Actual vs Predicted  (R2 = {knn_r2:.3f})')
axes[0].legend()

residuals_knn = y_test_knn - y_pred_knn
axes[1].scatter(y_pred_knn, residuals_knn, alpha=0.5, color=PALETTE[1],
                edgecolors='white', linewidth=0.4, s=40)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted (minutes)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.savefig('fig_phase4_knn_best.png', bbox_inches='tight')
plt.show()

### KNN Regression Interpretation

Best KNN (k=10): R2 = 0.77, MAE = 27 minutes. This is a dramatic improvement over linear regression (R2 = 0.05).

The k-tuning curve shows a sweet spot at k=10: k=1 overfits (memorises training data), while k=50 over-smooths and loses local structure.

The improvement from 0.05 to 0.77 comes from two changes: (1) adding time and prior-listening features beyond raw weather, and (2) allowing nonlinear relationships via the nearest-neighbor averaging mechanism.

**Key takeaway:** Listening duration is driven by complex nonlinear interactions between season, weekly routine, and prior listening patterns. Weather alone is insufficient.

---
## 4. Decision Tree Classification

**Question:** Can a decision tree infer the weather condition from listening behavior and time features alone?

**Features:** `n_plays`, `total_listen_min`, `skip_rate`, `fraction_complete`, `day_of_week`, `month`, `year`  
*(No weather features - tests whether listening implicitly encodes weather)*

**Target:** `condition` (4-class: Clear, Partly Cloudy, Rainy, Snowy)

Max depth is tuned to prevent overfitting.

In [ ]:
le_cond = LabelEncoder()
y_dt = le_cond.fit_transform(df['condition'])

# No weather features used - purely listening + time
X_dt = df[['n_plays', 'total_listen_min', 'skip_rate', 'fraction_complete',
           'day_of_week', 'month', 'year']]

X_train_dt, X_test_dt, y_train_dt, y_test_dt = train_test_split(
    X_dt, y_dt, test_size=0.2, random_state=42
)

depths = [2, 3, 4, 5, 7, 10, None]
dt_results = []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train_dt, y_train_dt)
    preds = dt.predict(X_test_dt)
    dt_results.append({'max_depth': str(d), 'Accuracy': accuracy_score(y_test_dt, preds)})

dt_df = pd.DataFrame(dt_results)
print('Decision Tree depth tuning:')
print(dt_df.to_string(index=False))
majority_baseline = pd.Series(y_test_dt).value_counts(normalize=True).max()
print(f'Majority class baseline: {majority_baseline:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Decision Tree: Depth Tuning', fontsize=13, fontweight='bold')

accs = [r['Accuracy'] for r in dt_results]
x_labels = [str(d) for d in depths]
axes[0].plot(range(len(depths)), accs, 'o-', color=PALETTE[2], linewidth=2, markersize=7)
axes[0].set_xticks(range(len(depths)))
axes[0].set_xticklabels(x_labels)
axes[0].axhline(majority_baseline, color='red', linestyle='--', linewidth=1, label='Baseline')
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy by max_depth')
axes[0].legend()
axes[0].grid(alpha=0.3)

cond_counts = df['condition'].value_counts()
axes[1].bar(cond_counts.index, cond_counts.values, color=PALETTE, edgecolor='white')
axes[1].set_xlabel('Weather Condition')
axes[1].set_ylabel('Days')
axes[1].set_title('Class Distribution')

plt.tight_layout()
plt.savefig('fig_phase4_dt_tuning.png', bbox_inches='tight')
plt.show()

In [ ]:
best_depth = 3
dt_best = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
dt_best.fit(X_train_dt, y_train_dt)
y_pred_dt = dt_best.predict(X_test_dt)

dt_acc = accuracy_score(y_test_dt, y_pred_dt)
print(f'Best Decision Tree (max_depth={best_depth}): Accuracy = {dt_acc:.4f}')
print()
print(classification_report(y_test_dt, y_pred_dt,
      target_names=le_cond.classes_, zero_division=0))
print('Feature importances:')
fi = pd.Series(dt_best.feature_importances_, index=X_dt.columns).sort_values(ascending=False)
print(fi.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Decision Tree (max_depth={best_depth}): Listening+Time to Weather Condition',
             fontsize=13, fontweight='bold')

cm_dt = confusion_matrix(y_test_dt, y_pred_dt)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=le_cond.classes_, yticklabels=le_cond.classes_)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

fi_sorted = fi.sort_values()
colors_fi = [PALETTE[0] if v > 0.1 else PALETTE[3] for v in fi_sorted]
fi_sorted.plot(kind='barh', ax=axes[1], color=colors_fi, edgecolor='white')
axes[1].set_xlabel('Importance')
axes[1].set_title('Feature Importances')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('fig_phase4_dt_results.png', bbox_inches='tight')
plt.show()

In [ ]:
# Tree structure visualisation
fig, ax = plt.subplots(figsize=(16, 6))
plot_tree(dt_best, feature_names=X_dt.columns, class_names=le_cond.classes_,
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title(f'Decision Tree Structure (max_depth={best_depth})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_phase4_dt_tree.png', bbox_inches='tight', dpi=150)
plt.show()

### Decision Tree Interpretation

Best accuracy = 49% (depth=3) on a 4-class problem with a majority baseline of ~50%. The model performs reasonably on Rainy and Partly Cloudy days but struggles with Clear and Snowy classes due to their scarcity (79 and 29 days respectively).

Feature importances reveal that month dominates at 53%, followed by year (15%) and total_listen_min (14%). Pure listening behavior features (n_plays, skip_rate) contribute very little.

**Key takeaway:** Time features encode weather condition far better than listening patterns do. Listening behavior is only weakly tied to the weather on any given day, confirming the asymmetry found in all previous methods.

---
## 5. Model Comparison & Summary

In [ ]:
# Summary table
summary = [
    {'Model': 'Linear Regression',   'Task': 'Regression',     'Metric': 'R2',        'Score': round(lr_r2, 4)},
    {'Model': 'Linear Regression',   'Task': 'Regression',     'Metric': 'MAE (min)', 'Score': round(lr_mae, 2)},
    {'Model': 'Logistic Regression', 'Task': 'Classification', 'Metric': 'Accuracy',  'Score': round(log_acc, 4)},
    {'Model': f'KNN (k={best_k})',   'Task': 'Regression',     'Metric': 'R2',        'Score': round(knn_r2, 4)},
    {'Model': f'KNN (k={best_k})',   'Task': 'Regression',     'Metric': 'MAE (min)', 'Score': round(knn_mae, 2)},
    {'Model': 'Decision Tree (d=3)', 'Task': 'Classification', 'Metric': 'Accuracy',  'Score': round(dt_acc, 4)},
]
print(pd.DataFrame(summary).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Phase 4: Model Performance Summary', fontsize=13, fontweight='bold')

# Regression R2
bars1 = axes[0].bar(['Linear\nRegression', f'KNN (k={best_k})'],
                    [lr_r2, knn_r2],
                    color=[PALETTE[3], PALETTE[0]], width=0.5, edgecolor='white')
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('R2 Score')
axes[0].set_title('Regression Models: R2 Comparison')
for bar, val in zip(bars1, [lr_r2, knn_r2]):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.3f}',
                 ha='center', va='bottom', fontweight='bold')

# Classification accuracy vs baseline
x = np.arange(2)
w = 0.35
baselines = [y_test_log.value_counts(normalize=True).max(),
             pd.Series(y_test_dt).value_counts(normalize=True).max()]
accs_cls  = [log_acc, dt_acc]
axes[1].bar(x - w/2, baselines, w, label='Baseline', color=PALETTE[3], alpha=0.7, edgecolor='white')
axes[1].bar(x + w/2, accs_cls,  w, label='Model',    color=PALETTE[2], edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Logistic\nRegression', 'Decision\nTree (d=3)'])
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Classification Models: Accuracy vs Baseline')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig_phase4_summary.png', bbox_inches='tight')
plt.show()

---
## 6. Conclusions

### What the ML methods reveal

| Finding | Evidence |
|---------|----------|
| Weather features have weak linear predictive power over listening time | Linear R2 = 0.05 |
| Listening behavior cannot predict rain | Logistic accuracy = 71% = baseline |
| Adding time + behavior features with a nonlinear model dramatically improves prediction | KNN R2 = 0.77 |
| Weather condition is encoded in time features, not listening patterns | DT feature importance: month = 53% |

### Overall narrative

Listening duration is predictable, but primarily from when in the year a day falls and from prior listening patterns - not from weather itself. The nonlinear KNN model (R2=0.77) shows that complex co-occurrence patterns between season, routine, and listening exist. Weather is not the primary driver.

### Limitations

- No Spotify audio features in this dataset (valence, energy). Including them could reveal richer   mood-based weather interactions.
- Class imbalance in Istanbul weather (72% non-rainy) makes binary rain classification inherently hard.
- Single individual: patterns reflect personal habits and cannot be generalised.

### Future work

- Apply Random Forest and Gradient Boosting (ensemble methods) for potentially higher R2.
- Use track-level Spotify audio features to model mood x weather interactions.
- Explore time-series models (LSTM, ARIMA) to capture sequential listening habits.
- Conduct clustering analysis to identify distinct 'listening day' personas.